# Reference implementation — regression check

The minimal round trip that is known to work: a button rendered from Almond,
clicks arriving in Scala, and Scala pushing state back to the browser.

**Run this first** whenever anything in the environment moves — a VS Code update,
an Almond upgrade, or an anywidget minor bump. It depends on nothing but `ujson`,
so it isolates "the stack still works" from "my code is wrong".

Keep it minimal. The handler here deliberately does not print and does not reply;
those are separate questions, and mixing them in would confuse the result.
See `probes.ipynb` for those.

## Cell 1 — open the comm and display the widget

In [ ]:
import $ivy.`com.lihaoyi::ujson:4.3.0`
import almond.interpreter.api.DisplayData
import java.nio.charset.StandardCharsets.UTF_8
import java.util.UUID

val received = scala.collection.mutable.ArrayBuffer.empty[String]

val esm = """
function render({ model, el }) {
  let count = () => model.get("count");
  let btn = document.createElement("button");
  btn.innerHTML = `count is ${count()}`;
  btn.addEventListener("click", () => {
    model.set("count", count() + 1);
    model.save_changes();
  });
  model.on("change:count", () => {
    btn.innerHTML = `count is ${count()}`;
  });
  el.appendChild(btn);
}
export default { render };
"""

val state = ujson.Obj(
  "_model_module"         -> "anywidget",
  "_model_module_version" -> "~0.11.*",
  "_model_name"           -> "AnyModel",
  "_view_module"          -> "anywidget",
  "_view_module_version"  -> "~0.11.*",
  "_view_name"            -> "AnyView",
  "_esm"                  -> esm,
  "count"                 -> 0
)

// Hold the id we generate. `sender` returns a `Comm` exposing only `message`
// and `close` — there is no id accessor on it in any published Almond, so the
// generated id is the only handle to the comm that exists.
val commId = UUID.randomUUID().toString

val _ = commHandler.sender(
  targetName = "jupyter.widget",
  id         = commId,
  data       = ujson.write(ujson.Obj("state" -> state, "buffer_paths" -> ujson.Arr())).getBytes(UTF_8),
  // Not optional. A missing or wrong version produces no widget at all, and the
  // only trace is a line in the webview console.
  metadata   = ujson.write(ujson.Obj("version" -> "2.0.0")).getBytes(UTF_8),
  onMessage  = (_, bytes) => received.synchronized {
                 received += new String(bytes, UTF_8)
               }
)

publish.display(DisplayData(Map(
  "application/vnd.jupyter.widget-view+json" -> ujson.write(ujson.Obj(
    "model_id" -> commId, "version_major" -> 2, "version_minor" -> 0
  ))
)))

## Cell 2 — confirm the clicks reached Scala

Click the button a few times, then run this.

**A working button proves nothing.** The anywidget frontend updates its own view
optimistically, before the kernel confirms anything. A button that increments on
click has demonstrated only that the JavaScript runs. This cell is the actual test.

In [ ]:
received.synchronized(received.toList).foreach(println)

Expected, one line per click:

```
{"method":"update","state":{"count":1},"buffer_paths":[]}
{"method":"update","state":{"count":2},"buffer_paths":[]}
{"method":"update","state":{"count":3},"buffer_paths":[]}
```

Nothing at all here, but a button that visibly incremented, means the frame never
left the browser. Check the websocket frames in the webview's Network tab.

## Cell 3 — push state from the kernel to the browser

The button should read `count is 99` without being touched.

In [ ]:
commHandler.commMessage(
  commId,
  ujson.write(ujson.Obj(
    "method"       -> "update",
    "state"        -> ujson.Obj("count" -> 99),
    "buffer_paths" -> ujson.Arr()
  )).getBytes(UTF_8),
  // Metadata is `{}` here: the protocol version goes on comm_open only.
  "{}".getBytes(UTF_8)
)

## If it does not work

Observation points, in order of usefulness:

1. **Webview console** — Command Palette, "Open Webview Developer Tools". Widget
   errors surface here and nowhere else. Filter the noise: a genuine widget error
   always has an `ipywidgets.js` or `ipywidgetsKernel.js` frame in its stack.
   `ERR Model is disposed!` with a `workbench.desktop.main.js` stack is a Monaco
   `TextModel`, not a widget model.
2. **Websocket frames** — Network tab, select the notebook socket. The protocol is
   JSON, so `comm_open`, `comm_msg` and `display_data` are directly readable. This
   answers the only question that matters when bisecting: *did the frame leave its
   origin?*
3. **Jupyter output channel** — View, Output, "Jupyter". Script-loading failures
   land here, e.g. `Script source for Widget anywidget@x.y.z not found`, which also
   reveals the exact version being requested from the CDN.